# 🚀 GIAI ĐOẠN 5 — KIẾN TRÚC STAIR5-v8.1 / BSCV: BOUNDED SOFT-CONSENSUS VIEW FOR WMSG-CSE
### 🏆 Hệ Thống Khuyến Nghị Đa Phương Thức Tiên Tiến (Multimodal Recommender System)
---
> **Bản quyền & Phát triển:** Nhóm Nghiên cứu KLTN — STAIR-Enhanced (HCMUS)  
> **Kiến trúc:** STAIR5-v8.1 (BSCV) — Bounded Soft-Consensus View for Weighted Multimodal Semantic Graph with Candidate Support Expansion  
> **Đặc tả Học thuật & Triển khai:** `docs/giai_doan_5/STAIR5_v8_1_Report.md` & `docs/giai_doan_5/STAIR5_v8_1_Implementation.md`  
> **Nguyên lý Cốt lõi & Đóng góp Đột phá:**
> 1. **Kế thừa Toàn vẹn V8 Backbone:** Bảo toàn nguyên vẹn không gian biểu diễn $d=64$ (và nhánh kiểm chứng $d=256$), các hàm mục tiêu BPR + Weighted NLGCL ($\lambda=0.01, \tau=0.2$), Candidate Support Expansion (CSE: $\eta=0.1, t=5.0, k=5$), bộ làm mịn hướng đa tần số (DF-BSC smoother: $S_8, \beta_3$), MI initialization, seen-item masking và validation-NDCG@20 checkpoint selection.
> 2. **Bounded Soft-Consensus View (BSCV):** Bổ sung view item phụ $J_i = I_i^1 + \rho_e g_i \kappa_i M_i$ cho duy nhất nhánh item-query $I^1 \to U^0$ của NLGCL trong quá trình huấn luyện; hoàn toàn không can thiệp vào inference, scoring, bộ tối ưu hay negative sampler.
> 3. **Fixed Support & Normalized Agreement:** Đồ thị consensus $C$ tái sử dụng chính xác fixed support CF đã có $\mathcal{E}_{CF}$, reweight bằng mức đồng thuận nội dung text/image $a_{ij} = (5a_{ij}^t + a_{ij}^v)/6$, chuẩn hóa đối xứng độc lập không thêm self-loop cho các đỉnh cô lập (zero isolated row gradient leak).
> 4. **Detached Norm Cap & Scalar Agreement Gate:** Khống chế tỷ lệ chuẩn $\kappa_i = \operatorname{stopgrad}\min(1, \frac{\|I_i^1\|_2}{\|M_i\|_2 + 10^{-8}})$ triệt tiêu nguy cơ gradient explosion; scalar agreement gate $g_i \in (0, 1)$ với mạng 3→8→1 SiLU (chỉ 41 tham số trainable) nhận context 3 chiều (cosine tách biệt, tỷ số chuẩn logarit, chỉ thị coverage).
> 5. **One-based Dose Schedule:** Lịch liều lượng theo epoch: $\rho_e = \rho_{\max} \operatorname{clip}(\frac{e-30}{30}, 0, 1)$ với $\rho_{\max} = 0.2$ (epochs 1–30: $\rho=0$, thuần V8; epoch 31: $\rho=0.2/30$; epoch 60: $\rho=0.2$).
> 6. **Phân lập Nhóm Tham Số (Parameter Group Isolation):** AdamWSEvo quản lý 3 nhóm tách biệt (User embeddings: LR 0.001; Item embeddings: LR 0.001 với BSC smoother; Learned gate: LR 0.0001, weight decay 0, không smoother, `zero_grad(set_to_none=True)` bảo toàn trạng thái khi liều lượng bằng 0).
---


## Cell 1 ⚙️ Thiết lập Môi trường, Dependencies & Đồng bộ Mã Nguồn STAIR5-v8.1 (BSCV)
- Tự động nhận diện môi trường (**Kaggle / Colab / Molab / Local**)
- Tự động **Clone / Fetch / Pull** mã nguồn mới nhất từ repository GitHub `ThanhChuong12/STAIR-Enhanced`
- **Cơ chế Tự Phục Hồi (Self-Healing Source Unpacker)**: Tự động trích xuất gói mã nguồn v8.1 hoàn chỉnh nếu git clone chưa kịp có commit mới nhất
- Thiết lập liên kết **Symlink & Copy 2 chiều** vào `/kaggle/working` đảm bảo mọi lệnh gọi trực tiếp đều tìm thấy file
- Cài đặt trọn bộ thư viện `freerec`, `torch-geometric`, `nvidia-ml-py`, `prettytable` với cơ chế `--no-deps` fallback chống xung đột `torchdata`
- Chạy báo cáo kiểm thử tiến trình con (**Subprocess Verification Report**) xác thực GPU, CUDA, FreeRec và STAIR5-v8.1


In [ ]:
# Cell 1: Môi trường, Dependencies, Tự Phục Hồi & Xác thực Subprocess
import os, shutil, subprocess, sys, glob, torch, base64, io, zipfile
from pathlib import Path

# Gói mã nguồn STAIR5-v8.1 tự phục hồi phòng ngừa trường hợp remote git chưa cập nhật commit
_STAIR5_V81_BUNDLE_B64 = "..."

# 1. Đồng bộ thư mục làm việc chuẩn tắc (tương thích Molab / Kaggle / Colab / Local)
_safe_home = os.path.expanduser('~')
_kw = '/kaggle/working'
os.makedirs(_kw, exist_ok=True)

# Tìm vị trí STAIR-Enhanced hiện hữu
_possible_stair_dirs = [
    os.path.join(_kw, 'STAIR-Enhanced'),
    os.path.join(_safe_home, 'STAIR-Enhanced'),
    os.path.abspath('STAIR-Enhanced'),
    os.path.abspath('.'),
]
_stair_dir = None
for _p in _possible_stair_dirs:
    if os.path.isdir(_p) and os.path.exists(os.path.join(_p, 'models', 'stair5_v8_1.py')):
        _stair_dir = _p
        break

if _stair_dir is None:
    _stair_dir = os.path.join(_kw, 'STAIR-Enhanced')
    if not os.path.exists(_stair_dir):
        print('📦 Đang clone repository STAIR-Enhanced (branch main)...')
        try:
            subprocess.run(
                ['git', 'clone', '--depth', '1', 'https://github.com/ThanhChuong12/STAIR-Enhanced.git', _stair_dir],
                check=True
            )
            print('✅ Clone thành công!')
        except Exception as _e:
            print(f'⚠️ Git clone gặp lỗi: {_e}')
            os.makedirs(_stair_dir, exist_ok=True)
    else:
        print('Thư mục STAIR-Enhanced đã tồn tại. Đang kiểm tra mã nguồn STAIR5-v8.1...')
        try:
            subprocess.run(['git', '-C', _stair_dir, 'pull', 'origin', 'main'], check=False, capture_output=True)
            print('✅ Đã kéo cập nhật mới nhất từ origin/main.')
        except Exception:
            pass

# Cơ chế Tự Phục Hồi (Self-Healing): Nếu models/stair5_v8_1.py chưa có, tự động dò quét Kaggle Input hoặc giải nén trực tiếp bundle
if not os.path.exists(os.path.join(_stair_dir, 'models', 'stair5_v8_1.py')):
    _input_matches = glob.glob('/kaggle/input/**/models/stair5_v8_1.py', recursive=True)
    if _input_matches:
        _src_repo = os.path.dirname(os.path.dirname(_input_matches[0]))
        print(f'📦 Phát hiện mã nguồn STAIR5-v8.1 từ Kaggle Input: {_src_repo} -> Sao chép vào {_stair_dir}...')
        os.makedirs(_stair_dir, exist_ok=True)
        shutil.copytree(_src_repo, _stair_dir, dirs_exist_ok=True)
    else:
        print('📦 Kích hoạt Self-Healing Unpacker: Đang giải nén gói mã nguồn STAIR5-v8.1...')
        try:
            _zip_bytes = base64.b64decode(_STAIR5_V81_BUNDLE_B64)
            with zipfile.ZipFile(io.BytesIO(_zip_bytes)) as _zf:
                _zf.extractall(_stair_dir)
            print('✅ Giải nén gói mã nguồn STAIR5-v8.1 thành công!')
        except Exception as _e:
            print(f'⚠️ Giải nén bundle cảnh báo: {_e}')

# Tạo liên kết symlink 2 chiều giữa ~/STAIR-Enhanced và /kaggle/working/STAIR-Enhanced
_home_stair = os.path.join(_safe_home, 'STAIR-Enhanced')
_kw_stair = os.path.join(_kw, 'STAIR-Enhanced')
if os.path.exists(_home_stair) and not os.path.exists(_kw_stair):
    try:
        os.symlink(_home_stair, _kw_stair)
    except Exception:
        pass
elif os.path.exists(_kw_stair) and not os.path.exists(_home_stair):
    try:
        os.symlink(_kw_stair, _home_stair)
    except Exception:
        pass

_active_dir = _stair_dir if os.path.exists(_stair_dir) else os.path.abspath('.')
os.chdir(_active_dir)

# Cầu nối đa tầng bảo đảm mọi thư mục con và tệp tin đều xuất hiện tại cả _stair_dir và /kaggle/working
for _item in ['main_stair5_v8_1.py', 'main_stair5_v8.py', 'main_stair6_v1.py', 'stair5_v8_1_config.py', 'configs', 'models', 'optimizers', 'tests', 'data', 'tools']:
    _src = os.path.join(_active_dir, _item)
    _dst = os.path.join(_kw, _item)
    if os.path.exists(_src):
        if os.path.isdir(_src):
            os.makedirs(_dst, exist_ok=True)
            for _subf in os.listdir(_src):
                _sf = os.path.join(_src, _subf)
                _df = os.path.join(_dst, _subf)
                if not os.path.exists(_df):
                    try:
                        os.symlink(_sf, _df)
                    except Exception:
                        try:
                            if os.path.isdir(_sf):
                                shutil.copytree(_sf, _df, dirs_exist_ok=True)
                            else:
                                shutil.copy2(_sf, _df)
                        except Exception:
                            pass
        else:
            if not os.path.exists(_dst):
                try:
                    os.symlink(_src, _dst)
                except Exception:
                    shutil.copy2(_src, _dst)

for _p in [_active_dir, _kw_stair, _home_stair, _kw, '.']:
    if _p and os.path.exists(_p) and (_p not in sys.path):
        sys.path.insert(0, _p)

_sub_env = os.environ.copy()
_sub_env['PYTHONPATH'] = os.pathsep.join([_p for _p in [_active_dir, _kw_stair, _home_stair, _kw, os.environ.get('PYTHONPATH', '')] if _p])
_sub_env['PYTHONWARNINGS'] = 'ignore'
_sub_env.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['PYTHONPATH'] = _sub_env['PYTHONPATH']
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

# Xóa cache module v8.1 cũ trong kernel nếu có
for _mod_name in list(sys.modules.keys()):
    if any(k in _mod_name for k in ['stair5_v8_1', 'models.stair5_v8_1', 'stair5_v8_1_config']):
        sys.modules.pop(_mod_name, None)

# 2. Cài đặt dependencies (bỏ qua torchdata vì Python 3.13 không hỗ trợ wheel, freerec_compat tự mock)
print('Cài đặt dependencies tương thích (freerec, torch-geometric, pyyaml, prettytable, kagglehub)...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q',
    'freerec>=0.9.7', 'torch-geometric', 'nvidia-ml-py', 'prettytable', 'matplotlib', 'pyyaml', 'scipy', 'pandas', 'kagglehub'
], check=False)

# Fallback cài đặt khẩn cấp --no-deps cho freerec nếu resolver gặp xung đột phiên bản torchdata
try:
    import freerec
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q', 'freerec>=0.9.7'], check=False)

# 3. Patch tương thích nội bộ PyTorch Dynamo / FreeRec
try:
    import models.freerec_compat
except Exception:
    try:
        import freerec_compat
    except Exception:
        pass

# 4. Xác minh môi trường trong tiến trình con
_verify_script = """
import torch, sys
print('=== SUBPROCESS ENVIRONMENT VERIFICATION (STAIR5-v8.1 BSCV) ===')
print('Python Executable :', sys.executable)
print('Python Version    :', sys.version.split()[0])
print('PyTorch Version   :', torch.__version__)
print('CUDA Available    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name       :', torch.cuda.get_device_name(0))
    print('Allocated VRAM    : {:.2f} MB'.format(torch.cuda.memory_allocated() / (1024**2)))
try:
    import freerec
    print('FreeRec Version   :', freerec.__version__)
except Exception as e:
    print('FreeRec Import    : FAILED ({})'.format(e))
try:
    from models.stair5_v8_1 import STAIR5_v8_1_Model
    from models.stair5_v8_1_consensus import ConsensusResidual, consensus_dose
    from models.stair5_v8_1_graph import build_consensus_graph
    from stair5_v8_1_config import validate_consensus_config, CONSENSUS_OPTIONS
    print('STAIR5-v8.1 BSCV  : VERIFIED OK (ConsensusResidual, Graph & Config validated)')
except Exception as e:
    print('STAIR5-v8.1 BSCV  : FAILED ({})'.format(e))
print('===============================================================')
"""
_res = subprocess.run([sys.executable, '-c', _verify_script], capture_output=True, text=True, env=_sub_env, cwd=_active_dir)
print(_res.stdout)
if _res.stderr and 'warning' not in _res.stderr.lower():
    print('Stderr:', _res.stderr)

print('✅ Khởi tạo môi trường STAIR5-v8.1 hoàn tất thành công.')


## Cell 2 📂 Chuẩn bị Dữ liệu (KaggleHub Auto-Fetch & Multi-Bridge sang /kaggle/data & Processed)
- Tự động dò quét dữ liệu thô và tiền xử lý (`.npy`, `.pkl`, `.txt`, `.inter`) trên toàn bộ các vị trí khả dĩ
- Tự động kích hoạt **KaggleHub** tải xuống `rainyle/stair-datasets-mmrec` nếu môi trường chưa có
- Khởi tạo **Cầu nối Symlink Đa Chiều (Multi-Bridge)** đảm bảo FreeRec tìm thấy tập dữ liệu tại `/kaggle/data`, `/kaggle/data/Processed`, và `STAIR-Enhanced/data`
- **Khắc phục triệt để lỗi Read-Only Filesystem**: Tạo thư mục thực (writable directory) cho từng tập dữ liệu và chủ động tạo thư mục `chunks/` độc lập để FreeRec ghi tập tin cache mà không bị xung đột với mount chỉ đọc của Kaggle Input


In [ ]:
# Cell 2: Chuẩn bị dữ liệu từ Kaggle Input / KaggleHub sang /kaggle/data & Processed
import os, shutil, glob

_DATA_ROOT = '/kaggle/data'
_PROCESSED_ROOT = os.path.join(_DATA_ROOT, 'Processed')
_LOCAL_DATA = '/kaggle/working/STAIR-Enhanced/data'
_LOCAL_PROCESSED = os.path.join(_LOCAL_DATA, 'Processed')
_HOME_DATA = os.path.expanduser('~/STAIR-Enhanced/data')
_HOME_PROCESSED = os.path.join(_HOME_DATA, 'Processed')

for _d in [_DATA_ROOT, _PROCESSED_ROOT, _LOCAL_DATA, _LOCAL_PROCESSED, _HOME_DATA, _HOME_PROCESSED]:
    os.makedirs(_d, exist_ok=True)

_TARGET_DATASETS = {
    'sports':      ('Amazon2014Sports_550_MMRec', ['sport', 'sports', 'amazon2014sports']),
    'baby':        ('Amazon2014Baby_550_MMRec', ['baby', 'amazon2014baby']),
    'electronics': ('Amazon2014Electronics_550_MMRec', ['electronic', 'electronics', 'amazon2014electronics']),
}

_REQUIRED_EXTENSIONS = ('.npy', '.pkl', '.txt', '.inter', '.item', '.pt', '.csv', '.yaml', '.json', '.jsonl')

def _bridge_directories(src_dir, target_folder):
    _destinations = [
        os.path.join(_DATA_ROOT, target_folder),
        os.path.join(_PROCESSED_ROOT, target_folder),
        os.path.join(_LOCAL_DATA, target_folder),
        os.path.join(_LOCAL_PROCESSED, target_folder),
        os.path.join(_HOME_DATA, target_folder),
        os.path.join(_HOME_PROCESSED, target_folder),
    ]
    for _dst in _destinations:
        if os.path.abspath(src_dir) == os.path.abspath(_dst):
            continue
        os.makedirs(_dst, exist_ok=True)
        # Tạo sẵn thư mục chunks writable để tránh lỗi khi đọc Kaggle Input read-only
        os.makedirs(os.path.join(_dst, 'chunks'), exist_ok=True)
        for _item in os.listdir(src_dir):
            _s_item = os.path.join(src_dir, _item)
            _d_item = os.path.join(_dst, _item)
            if os.path.isdir(_s_item):
                if not os.path.exists(_d_item):
                    try:
                        os.symlink(_s_item, _d_item)
                    except Exception:
                        shutil.copytree(_s_item, _d_item, dirs_exist_ok=True)
            else:
                if not os.path.exists(_d_item) or os.path.getsize(_d_item) == 0:
                    try:
                        os.symlink(_s_item, _d_item)
                    except Exception:
                        shutil.copy2(_s_item, _d_item)

def _find_existing_dataset(folder_name, aliases):
    _search_bases = ['/kaggle/input', '/kaggle/data', '.', '..', '/kaggle/working', os.path.expanduser('~')]
    for _base in _search_bases:
        if not os.path.exists(_base):
            continue
        for _root, _dirs, _files in os.walk(_base):
            _bname = os.path.basename(_root).lower()
            if _bname == folder_name.lower() or any(_alias in _bname for _alias in aliases):
                if any(_f.endswith(_REQUIRED_EXTENSIONS) for _f in _files):
                    return _root
    return None

print('=' * 80)
print('TIẾN TRÌNH DÒ QUÉT & LIÊN KẾT DỮ LIỆU TỰ ĐỘNG CHO STAIR5-v8.1 (BSCV):')

_missing = []
for _key, (_folder, _aliases) in _TARGET_DATASETS.items():
    if not _find_existing_dataset(_folder, _aliases):
        _missing.append(_key)

if _missing:
    print(f'📦 Phát hiện thiếu {_missing}. Đang kích hoạt KaggleHub auto-download...')
    try:
        import kagglehub
        _token = os.environ.get('KAGGLE_API_TOKEN', 'KGAT_4d9ab5bae4040ef4178b379d0262131b')
        _kaggle_cfg_dir = os.path.expanduser('~/.kaggle')
        os.makedirs(_kaggle_cfg_dir, exist_ok=True)
        with open(os.path.join(_kaggle_cfg_dir, 'access_token'), 'w') as _tf:
            _tf.write(_token)
        try:
            os.chmod(os.path.join(_kaggle_cfg_dir, 'access_token'), 0o600)
        except Exception:
            pass

        _slug = 'rainyle/stair-datasets-mmrec'
        print(f'  ↳ Đang tải dataset {_slug}...')
        _dl_path = kagglehub.dataset_download(_slug)
        print(f'  ↳ Tải thành công tại: {_dl_path}')

        os.makedirs('/kaggle/input', exist_ok=True)
        _in_symlink = '/kaggle/input/stair-datasets-mmrec'
        if not os.path.exists(_in_symlink) and not os.path.islink(_in_symlink):
            try:
                os.symlink(_dl_path, _in_symlink)
            except Exception:
                pass

        for _sub in os.listdir(_dl_path):
            _sub_full = os.path.join(_dl_path, _sub)
            _alias_in = os.path.join('/kaggle/input', _sub)
            if not os.path.exists(_alias_in) and not os.path.islink(_alias_in):
                try:
                    os.symlink(_sub_full, _alias_in)
                except Exception:
                    pass
    except Exception as _e:
        print(f'⚠️ KaggleHub auto-download cảnh báo: {_e}')

_prepared_data = set()
for _key, (_folder_name, _aliases) in _TARGET_DATASETS.items():
    _src = _find_existing_dataset(_folder_name, _aliases)
    if _src:
        _bridge_directories(_src, _folder_name)
        _prepared_data.add(_key)
        _cnt = len(os.listdir(_src))
        print(f'  ✅ [SẴN SÀNG] {_key.upper():12s} -> Nguồn: {_src} ({_cnt} files)')
    else:
        print(f'  ⚠️ [CHƯA THẤY] {_key.upper():12s} -> Sẽ kiểm tra lại thư mục nguồn.')
print('=' * 80)


## Cell 3 🧪 Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v8.1 BSCV
- Thực thi nghiệm thu toàn bộ các bài kiểm thử tự động của STAIR5-v8.1 trước khi huấn luyện:
  1. `test_stair5_v8_1_consensus.py` (Algebraic contracts, norm cap, scalar gate 41 params, seed isolation, autograd flow)
  2. `test_stair5_v8_1_graph.py` (CF support reuse, semantic weighting, sparse normalization, cache integrity, no dense catalog operations)
  3. `test_stair5_v8_1_pipeline.py` (Parameter group isolation, exact V8 recovery at $\rho=0$, duplicate-ID expansion, checkpoint continuation)


In [ ]:
# Cell 3: Chạy Bộ Kiểm Thử Độc Lập Toàn Diện Cho STAIR5-v8.1 BSCV
import os, sys, subprocess

print('=' * 80)
print('🚀 CHẠY BỘ KIỂM THỬ ĐỘC LẬP TOÀN DIỆN CHO STAIR5-v8.1 BSCV...')
print('=' * 80)

_test_cwd = _stair_dir if ('_stair_dir' in globals() and _stair_dir and os.path.exists(_stair_dir)) else (
    '/kaggle/working/STAIR-Enhanced' if os.path.exists('/kaggle/working/STAIR-Enhanced') else '.'
)

# Thực thi các bài kiểm thử đơn vị độc lập, bỏ qua subprocess long-timeout test
_test_cmd = [
    sys.executable, '-m', 'pytest',
    os.path.join(_test_cwd, 'tests', 'test_stair5_v8_1_consensus.py'),
    os.path.join(_test_cwd, 'tests', 'test_stair5_v8_1_graph.py'),
    os.path.join(_test_cwd, 'tests', 'test_stair5_v8_1_pipeline.py'),
    '-k', 'not test_actual_entrypoint',
    '-v'
]

_sub_env = os.environ.copy()
_sub_env['PYTHONPATH'] = os.pathsep.join([_p for _p in ['.', _test_cwd, '/kaggle/working', os.environ.get('PYTHONPATH', '')] if _p])

_res = subprocess.run(_test_cmd, capture_output=True, text=True, env=_sub_env, cwd=_test_cwd)
print(_res.stdout)
if _res.stderr:
    print(_res.stderr)
if _res.returncode == 0:
    print('✅ TOÀN BỘ CÁC UNIT TESTS CHO STAIR5-v8.1 ĐÃ VƯỢT QUA XUẤT SẮC!')
else:
    print(f'⚠️ Cảnh báo kiểm thử: Mã trả về {_res.returncode}')


## Cell 4 🛠️ Telemetry Engine: Training Runner, GPU Hardware Profiler & Visualizer
- Quản lý tiểu trình chạy ngầm **NVML / PyTorch VRAM Polling Thread** (chu kỳ ~2s) ghi nhận đỉnh tiêu thụ bộ nhớ
- Quản lý tiến trình con `run_training_stair5_v8_1()` với cơ chế **Live Log Streaming** trực tiếp lên giao diện Notebook
- Hàm phân tích log chuẩn tắc: `extract_test_metrics`, `extract_best_validation`, `parse_training_losses`, `parse_telemetry_jsonl`
- **Consensus Telemetry Parser**: Trích xuất chi tiết `consensus_telemetry.jsonl` (liều lượng $\rho_e$, `gate_mean`, tỷ lệ kích hoạt norm cap, độ lệch góc cosine `shift_cos_mean`) phục vụ phân tích khoa học chuyên sâu
- Tự động khắc phục đường dẫn file runner và tệp tin YAML cấu hình trên mọi cấu trúc đĩa Kaggle


In [ ]:
# Cell 4: Telemetry Engine — Training Runner, Hardware Profiler & Visualization
import os, sys, time, re, json, threading, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.stdout.reconfigure(encoding='utf-8') if hasattr(sys.stdout, 'reconfigure') else None

TRACKED_METRICS = ['Recall@10', 'Recall@20', 'NDCG@10', 'NDCG@20']

# Bảng đối chuẩn đa thế hệ lịch sử trên cả 3 tập dữ liệu
BASELINE_REF = {
    'sports':      {'Recall@10': 0.0743, 'Recall@20': 0.1111, 'NDCG@10': 0.0405, 'NDCG@20': 0.0500},
    'baby':        {'Recall@10': 0.0674, 'Recall@20': 0.1042, 'NDCG@10': 0.0359, 'NDCG@20': 0.0454},
    'electronics': {'Recall@10': 0.0442, 'Recall@20': 0.0665, 'NDCG@10': 0.0246, 'NDCG@20': 0.0303},
}

V4_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0678, 'NDCG@10': 0.0260, 'NDCG@20': 0.0316},
}

V6_REF = {
    'sports':      {'Recall@10': 0.0765, 'Recall@20': 0.1143, 'NDCG@10': 0.0419, 'NDCG@20': 0.0517, 'epoch': 485},
    'baby':        {'Recall@10': 0.0677, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@10': 0.0458, 'Recall@20': 0.0681, 'NDCG@10': 0.0259, 'NDCG@20': 0.0317, 'epoch': 440},
}

V7_REF = {
    'sports':      {'Recall@10': 0.0768, 'Recall@20': 0.1147, 'NDCG@10': 0.0422, 'NDCG@20': 0.0520, 'epoch': 490},
    'baby':        {'Recall@10': 0.0678, 'Recall@20': 0.1056, 'NDCG@10': 0.0364, 'NDCG@20': 0.0461, 'epoch': 480},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0684, 'NDCG@10': 0.0262, 'NDCG@20': 0.0318, 'epoch': 485},
}

# Kết quả chính thức của V8 (WMSG-CSE) đối chứng từ docs/giai_doan_5/STAIR5_v8_1_Report.md
V8_REF = {
    'sports':      {'Recall@10': 0.0762, 'Recall@20': 0.1142, 'NDCG@10': 0.0420, 'NDCG@20': 0.0518, 'epoch': 500},
    'baby':        {'Recall@10': 0.0675, 'Recall@20': 0.1054, 'NDCG@10': 0.0365, 'NDCG@20': 0.0462, 'epoch': 430},
    'electronics': {'Recall@10': 0.0460, 'Recall@20': 0.0680, 'NDCG@10': 0.0260, 'NDCG@20': 0.0317, 'epoch': 500},
}

DATASET_PROFILES = {
    'sports': {
        'name':        'Amazon Sports',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '35,598 Users | 18,357 Items | 296K Interactions',
        'color':       '#ff7f0e',
        'approx_mins': 45.0,
    },
    'baby': {
        'name':        'Amazon Baby',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '19,445 Users | 7,050 Items | 160K Interactions',
        'color':       '#1f77b4',
        'approx_mins': 20.0,
    },
    'electronics': {
        'name':        'Amazon Electronics',
        'domain':      'E-commerce (Visual + Textual)',
        'scale':       '192,403 Users | 63,001 Items | 1.68M Interactions',
        'color':       '#2ca02c',
        'approx_mins': 165.0,
    },
}

STAIR5_V8_1_RESULTS = {}
vram_profile = {}

def resolve_path(p):
    if not p: return ''
    if os.path.isabs(p) and os.path.exists(p): return p
    kw_cand = os.path.join('/kaggle/working', p.lstrip('/'))
    if os.path.exists(kw_cand): return kw_cand
    stair_cand = os.path.join('/kaggle/working/STAIR-Enhanced', p.lstrip('/'))
    if os.path.exists(stair_cand): return stair_cand
    home_cand = os.path.expanduser(os.path.join('~/STAIR-Enhanced', p.lstrip('/')))
    if os.path.exists(home_cand): return home_cand
    return p

def vram_monitor(key, stop_event, interval_sec=2.0):
    samples = []
    has_nvml = False
    handle = None
    try:
        import pynvml
        pynvml.nvmlInit()
        handle = pynvml.nvmlDeviceGetHandleByIndex(0)
        has_nvml = True
    except Exception:
        pass

    while not stop_event.is_set():
        if has_nvml and handle:
            try:
                info = pynvml.nvmlDeviceGetMemoryInfo(handle)
                samples.append(info.used / (1024**2))
            except Exception:
                pass
        elif torch.cuda.is_available():
            try:
                samples.append(torch.cuda.memory_allocated() / (1024**2))
            except Exception:
                pass
        time.sleep(interval_sec)

    vram_profile[key] = samples
    if has_nvml:
        try:
            pynvml.nvmlShutdown()
        except Exception:
            pass

def parse_telemetry_jsonl(artifact_dir):
    artifact_dir = resolve_path(artifact_dir)
    jsonl_path = os.path.join(artifact_dir, 'training_telemetry.jsonl')
    records = []
    if os.path.exists(jsonl_path):
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        pass
    return records

def parse_consensus_telemetry_jsonl(artifact_dir):
    artifact_dir = resolve_path(artifact_dir)
    jsonl_path = os.path.join(artifact_dir, 'consensus_telemetry.jsonl')
    records = []
    if os.path.exists(jsonl_path):
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        records.append(json.loads(line))
                    except Exception:
                        pass
    return records

def extract_best_validation(log_path):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = r'VALID\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)'
    best_val_epoch = None
    best_val_score = -1.0
    best_val_metrics = {}
    for m in re.finditer(pattern, content, re.DOTALL):
        ep = int(m.group(1))
        snippet = m.group(2)
        ndcg_m = re.search(r'NDCG@20(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', snippet, re.IGNORECASE)
        if ndcg_m:
            score = float(ndcg_m.group(1))
            if score > best_val_score:
                best_val_score = score
                best_val_epoch = ep
                best_val_metrics = {}
                for metric in TRACKED_METRICS:
                    met_m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', snippet, re.IGNORECASE)
                    if met_m:
                        best_val_metrics[metric] = float(met_m.group(1))
    return best_val_epoch, best_val_metrics

def extract_test_metrics(log_path, artifact_dir=None):
    if artifact_dir:
        sel_json = os.path.join(resolve_path(artifact_dir), 'selected_test_metrics.json')
        if os.path.exists(sel_json):
            try:
                with open(sel_json, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                return data.get('best_epoch'), data.get('test_metrics', {})
            except Exception:
                pass
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return None, {}
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    best_matches = list(re.finditer(r'\[Coach\]\s*>>>\s*Load best model\s*@Epoch:\s*(\d+)', content))
    best_match = best_matches[-1] if best_matches else None
    if not best_match:
        return None, {}
    content_after = content[best_match.end():]
    test_match = re.search(r'TEST\s+@Epoch:\s*(\d+)(.*?)(?:=================|$)', content_after, re.DOTALL)
    if not test_match:
        return None, {}
    test_epoch = int(test_match.group(1))
    test_snippet = test_match.group(2)
    test_metrics = {}
    for metric in TRACKED_METRICS:
        m = re.search(rf'{metric}(?:\s*Avg)?\s*[:\s]+\s*([0-9.]+)', test_snippet, re.IGNORECASE)
        if m:
            test_metrics[metric] = float(m.group(1))
    return test_epoch, test_metrics

def parse_training_losses(log_path, telemetry=None):
    if telemetry and len(telemetry) > 0:
        bpr_losses = [(r['epoch'], float(r['bpr_loss'])) for r in telemetry if 'bpr_loss' in r]
        cl_losses = [(r['epoch'], float(r['cl_loss'])) for r in telemetry if 'cl_loss' in r]
        total_losses = [(r['epoch'], float(r['total_loss'])) for r in telemetry if 'total_loss' in r]
        return bpr_losses, cl_losses, total_losses
    log_path = resolve_path(log_path)
    bpr_losses, cl_losses, total_losses = [], [], []
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
        matches = re.findall(r'LOSS\s+Avg:\s*([0-9.]+)', content)
        for i, val in enumerate(matches):
            total_losses.append((i + 1, float(val)))
    return bpr_losses, cl_losses, total_losses

def parse_valid_metric(log_path, metric='NDCG@20'):
    log_path = resolve_path(log_path)
    if not os.path.exists(log_path):
        return []
    with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    pattern = rf'VALID\s+@Epoch:\s*(\d+).*?{metric}\s+Avg:\s*([0-9.]+)'
    matches = re.findall(pattern, content, re.IGNORECASE)
    return [(int(ep), float(v)) for ep, v in matches]

def run_training_stair5_v8_1(key, config_yaml=None, data_root=None, log_path=None, artifact_dir=None, **kwargs):
    """Executes STAIR5-v8.1 BSCV training with real-time streaming and fail-fast handling."""
    cfg_dict = STAIR5_V8_1_CONFIGS.get(key, {}).copy()
    cfg_dict.update(kwargs)

    # 1. Tìm file runner main_stair5_v8_1.py tuyệt đối
    _possible_runners = [
        os.path.join(_active_dir, 'main_stair5_v8_1.py') if '_active_dir' in globals() else '',
        os.path.join(_stair_dir, 'main_stair5_v8_1.py') if '_stair_dir' in globals() else '',
        '/kaggle/working/STAIR-Enhanced/main_stair5_v8_1.py',
        '/kaggle/working/main_stair5_v8_1.py',
        os.path.expanduser('~/STAIR-Enhanced/main_stair5_v8_1.py'),
        'main_stair5_v8_1.py',
    ]
    runner_py = next((r for r in _possible_runners if r and os.path.exists(r)), 'main_stair5_v8_1.py')
    _active_dir_resolved = os.path.dirname(os.path.abspath(runner_py)) if os.path.exists(runner_py) else (
        _active_dir if '_active_dir' in globals() else '.'
    )

    # 2. Tìm file config yaml tuyệt đối
    config_yaml = config_yaml or cfg_dict.get('yaml')
    if not os.path.exists(config_yaml):
        cand_y1 = os.path.join(_active_dir_resolved, 'configs', os.path.basename(config_yaml))
        cand_y2 = os.path.join('/kaggle/working/STAIR-Enhanced/configs', os.path.basename(config_yaml))
        cand_y3 = os.path.join('/kaggle/working/configs', os.path.basename(config_yaml))
        cand_y4 = os.path.expanduser(f'~/STAIR-Enhanced/configs/{os.path.basename(config_yaml)}')
        for cand in [cand_y1, cand_y2, cand_y3, cand_y4]:
            if os.path.exists(cand):
                config_yaml = cand
                break

    label = f"{key}_v81_{cfg_dict.get('consensus_mode', 'soft')}_seed{cfg_dict.get('seed', 1)}"
    log_path = log_path or str(Path(cfg_dict['log']).parent / (label + '.log'))
    artifact_dir = artifact_dir or str(Path(cfg_dict['artifact_dir']).parent / label)
    resuming = bool(cfg_dict.get('resume_from'))
    if not resuming and (Path(log_path).exists() or Path(artifact_dir).exists()):
        print(f'⚠️ Thông báo: Ghi đè hoặc tái lập kịch bản run cho {label}.')
        if Path(artifact_dir).exists():
            for stale in ('manifest.json', 'training_checkpoint.pt', 'training_telemetry.jsonl', 'consensus_telemetry.jsonl', 'best_model.pt', 'last_model.pt', 'selected_test_metrics.json'):
                sp_file = Path(artifact_dir) / stale
                if sp_file.exists():
                    try:
                        sp_file.unlink()
                    except OSError:
                        pass
    STAIR5_V8_1_CONFIGS[key].update(log=log_path, artifact_dir=artifact_dir, yaml=config_yaml)

    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    os.makedirs(artifact_dir, exist_ok=True)

    print('=' * 80)
    print(f'🚀 KHỞI ĐỘNG TIẾN TRÌNH HUẤN LUYỆN STAIR5-v8.1 / BSCV: {key.upper()}')
    print(f'  * Dataset Key         : {key}')
    print(f'  * Runner Executable   : {runner_py}')
    print(f'  * YAML Configuration  : {config_yaml}')
    print(f'  * Log Path            : {log_path}')
    print(f'  * Artifact Directory  : {artifact_dir}')
    print(f"  * Consensus Mode      : {cfg_dict.get('consensus_mode', 'soft')}")
    print(f"  * Consensus Gate      : {cfg_dict.get('consensus_gate', 'learned')}")
    print(f"  * Consensus Max Rho   : {cfg_dict.get('consensus_rho', 0.2)}")
    print(f"  * Consensus Warmup    : {cfg_dict.get('consensus_warmup', 30)}")
    print(f"  * Consensus Ramp      : {cfg_dict.get('consensus_ramp', 30)}")
    print(f"  * Consensus Floor     : {cfg_dict.get('consensus_floor', 0.05)}")
    print(f"  * Consensus Layout    : {cfg_dict.get('consensus_sparse_layout', 'auto')}")
    print(f"  * Operator Blend (η)  : {cfg_dict.get('eta', 0.1)}")
    print(f"  * Seed                : {cfg_dict.get('seed', 1)}")
    print('=' * 80)

    stop_evt = threading.Event()
    th = threading.Thread(target=vram_monitor, args=(key, stop_evt), daemon=True)
    th.start()

    t0 = time.time()

    # Xác định data_root phù hợp
    if data_root is None:
        data_root = '/kaggle/data' if os.path.exists('/kaggle/data') else 'data'

    device_str = str(cfg_dict.get('device', '0' if torch.cuda.is_available() else 'cpu'))
    cmd = [
        sys.executable, '-u', str(runner_py),
        '--config', str(config_yaml),
        '--root', str(data_root),
        '--device', str(device_str),
        '--ranking', 'full',
        '--artifact-dir', str(artifact_dir),
    ]

    defaults = {
        'v8_arm': 'WMSG-core', 'semantic_mode': 'weighted', 'edge_power': 2.0, 'edge_floor': 0.05,
        'semantic_mix': 1.0, 'alignment': 'off', 'anchor_seed': 2718, 'anchor_per_stratum': 1024,
        'relation_gate': 'off', 'relation_strength': 0.0, 'relation_threshold': 0.7,
        'relation_temperature': 0.1, 'bsc_residual': 0.0, 'lambda_dirichlet': 0.0,
        'seed': 1, 'eval_freq': 5, 'eta': 0.1, 'k_cf': 5, 'c_min': 2, 't_shrinkage': 5.0,
        'cf_block_size': 64, 'cf_memory_budget_mib': 128,
        'lambda_nlgcl': 0.01, 'nlgcl_tau': 0.2, 'nlgcl_G': 1, 'nlgcl_alpha': 0.5, 'cl_chunk_size': 1024,
        'knn_chunk_size': 1024, 'knn_device': 'cpu',
        'graph_cache_dir': '/kaggle/working/runs/stair5_v81/graph_cache',
        # Consensus options
        'consensus_mode': 'soft', 'consensus_gate': 'learned', 'consensus_rho': 0.2,
        'consensus_warmup': 30, 'consensus_ramp': 30, 'consensus_floor': 0.05,
        'consensus_text_weight': 5.0 / 6.0, 'consensus_hidden': 8, 'consensus_eps': 1e-8,
        'consensus_seed': 2718, 'consensus_placebo_seed': 1, 'consensus_memory_budget_mib': 64.0,
        'consensus_sparse_layout': 'auto', 'consensus_diagnostic_interval': 25, 'lr_aux_scale': 0.1,
    }
    settings = {**defaults, **cfg_dict}
    for k_item in defaults:
        cmd += ['--' + k_item.replace('_', '-'), str(settings[k_item])]
    for k_item in ('epochs', 'batch_size', 'gamma', 'lr', 'weight_decay', 'num_workers'):
        if k_item in cfg_dict:
            cmd += ['--' + k_item.replace('_', '-'), str(cfg_dict[k_item])]
    if cfg_dict.get('resume_from'):
        cmd += ['--resume-from', str(cfg_dict['resume_from'])]

    print('Command:', ' '.join(cmd))
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([p for p in ['.', _active_dir_resolved, '/kaggle/working', '/kaggle/working/STAIR-Enhanced', os.environ.get('PYTHONPATH', '')] if p])
    env['PYTHONUTF8'] = '1'

    try:
        with open(log_path, 'a' if resuming else 'w', encoding='utf-8') as lf:
            proc = subprocess.Popen(
                cmd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                encoding='utf-8', errors='replace',
                bufsize=1,
                env=env,
                cwd=_active_dir_resolved,
            )
            for line in proc.stdout:
                sys.stdout.write(line)
                sys.stdout.flush()
                lf.write(line)
                lf.flush()
            proc.wait()
            ret_code = proc.returncode
    finally:
        stop_evt.set()
        th.join(timeout=3.0)

    elapsed = time.time() - t0
    print(f'⏱️ Thời gian thực thi ({key.upper()}): {elapsed/60:.2f} phút | Trạng thái exit code: {ret_code}')
    if ret_code != 0:
        raise RuntimeError(f'Tiến trình huấn luyện {key.upper()} thất bại với mã lỗi {ret_code}. Kiểm tra log: {log_path}')
    return elapsed


## Cell 5 📋 Cấu hình Siêu tham số STAIR5-v8.1 BSCV (Dataset-Adaptive Matrix & Blackwell Tuning)
- Ma trận siêu tham số thích ứng theo từng tập dữ liệu (`Amazon Sports`, `Amazon Baby`, `Amazon Electronics`)
- Kích hoạt cơ chế Soft Consensus (`consensus_mode='soft'`, `consensus_gate='learned'`) với lịch liều lượng $30+30$ (`consensus_warmup=30`, `consensus_ramp=30`)
- Tự động nhận diện GPU dung lượng lớn (**NVIDIA Blackwell / RTX PRO 6000 96GB**) để mở rộng `cl_chunk_size` tối ưu tốc độ, trong khi giữ nguyên giới hạn an toàn 128 trên Tesla T4 16GB


In [ ]:
# Cell 5: Cấu hình Siêu tham số STAIR5-v8.1 BSCV (Dataset-Adaptive Matrix)
import os, uuid, torch
from datetime import datetime
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6]

for _d in ['/kaggle/working/logs/stair5_v81', '/kaggle/working/reports', '/kaggle/working/runs/stair5_v81/graph_cache',
           os.path.expanduser('~/logs/stair5_v81'), os.path.expanduser('~/reports')]:
    try:
        os.makedirs(_d, exist_ok=True)
    except Exception:
        pass

# Nhận diện GPU VRAM: Nếu > 32GB VRAM (Blackwell RTX PRO 6000 96GB), tự động mở rộng chunk size để đạt hiệu năng tối đa
_is_blackwell = torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory > 32 * (1024**3)
_elec_cl_chunk = 1024 if _is_blackwell else 128

if _is_blackwell:
    print(f'🚀 PHÁT HIỆN GPU DUNG LƯỢNG LỚN: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB VRAM).')
    print(f'  ↳ Tối ưu Electronics: cl_chunk_size={_elec_cl_chunk} (Tăng tốc tối đa).')

STAIR5_V8_1_CONFIGS = {
    'sports': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Sports_STAIR5_v8_1.yaml',
        'log':                f'/kaggle/working/logs/stair5_v81/{RUN_ID}/sports_stair5_v8_1.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v81/{RUN_ID}/sports_BSCV_seed1',
        'v8_arm':             'WMSG-core',
        'semantic_mode':      'weighted',
        'edge_power':         2.0,
        'edge_floor':         0.05,
        'semantic_mix':       1.0,
        'alignment':          'off',
        'relation_gate':      'off',
        'relation_strength':  0.0,
        'bsc_residual':       0.0,
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'consensus_mode':     'soft',
        'consensus_gate':     'learned',
        'consensus_rho':      0.2,
        'consensus_warmup':   30,
        'consensus_ramp':     30,
        'consensus_floor':    0.05,
        'consensus_text_weight': 5.0 / 6.0,
        'consensus_sparse_layout': 'auto',
        'consensus_diagnostic_interval': 25,
        'lr_aux_scale':       0.1,
        'seed':               1,
        'eval_freq':          5,
    },
    'baby': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Baby_STAIR5_v8_1.yaml',
        'log':                f'/kaggle/working/logs/stair5_v81/{RUN_ID}/baby_stair5_v8_1.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v81/{RUN_ID}/baby_BSCV_seed1',
        'v8_arm':             'WMSG-core',
        'semantic_mode':      'weighted',
        'edge_power':         2.0,
        'edge_floor':         0.05,
        'semantic_mix':       1.0,
        'alignment':          'off',
        'relation_gate':      'off',
        'relation_strength':  0.0,
        'bsc_residual':       0.0,
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      1024,
        'consensus_mode':     'soft',
        'consensus_gate':     'learned',
        'consensus_rho':      0.2,
        'consensus_warmup':   30,
        'consensus_ramp':     30,
        'consensus_floor':    0.05,
        'consensus_text_weight': 5.0 / 6.0,
        'consensus_sparse_layout': 'auto',
        'consensus_diagnostic_interval': 25,
        'lr_aux_scale':       0.1,
        'seed':               1,
        'eval_freq':          5,
    },
    'electronics': {
        'yaml':               '/kaggle/working/STAIR-Enhanced/configs/Amazon2014Electronics_STAIR5_v8_1.yaml',
        'log':                f'/kaggle/working/logs/stair5_v81/{RUN_ID}/electronics_stair5_v8_1.log',
        'artifact_dir':       f'/kaggle/working/runs/stair5_v81/{RUN_ID}/electronics_BSCV_seed1',
        'v8_arm':             'WMSG-core',
        'semantic_mode':      'weighted',
        'edge_power':         2.0,
        'edge_floor':         0.05,
        'semantic_mix':       1.0,
        'alignment':          'off',
        'relation_gate':      'off',
        'relation_strength':  0.0,
        'bsc_residual':       0.0,
        'eta':                0.1,
        'k_cf':               5,
        'c_min':              2,
        't_shrinkage':        5.0,
        'cf_block_size':      64,
        'lambda_nlgcl':       0.01,
        'nlgcl_tau':          0.2,
        'nlgcl_G':            1,
        'nlgcl_alpha':        0.5,
        'cl_chunk_size':      _elec_cl_chunk,
        'consensus_mode':     'soft',
        'consensus_gate':     'learned',
        'consensus_rho':      0.2,
        'consensus_warmup':   30,
        'consensus_ramp':     30,
        'consensus_floor':    0.05,
        'consensus_text_weight': 5.0 / 6.0,
        'consensus_sparse_layout': 'auto',
        'consensus_diagnostic_interval': 25,
        'lr_aux_scale':       0.1,
        'seed':               1,
        'eval_freq':          5,
    },
}

print('✅ Ma trận cấu hình STAIR5-v8.1 BSCV đã nạp thành công:')
for _k, _v in STAIR5_V8_1_CONFIGS.items():
    print(f"  * {_k.upper():12s} -> Mode: {_v['consensus_mode']} | Gate: {_v['consensus_gate']} | MaxRho: {_v['consensus_rho']} | Warmup: {_v['consensus_warmup']} | Config: {os.path.basename(_v['yaml'])}")


## Cell 6a 🏋️ Huấn luyện Pha A — Amazon Sports (Pilot STAIR5-v8.1 / BSCV)
- Khởi động huấn luyện STAIR5-v8.1 trên tập dữ liệu Amazon Sports (~296K tương tác, 500 epochs)
- Kích hoạt 30 epochs warmup (rho=0, thuần V8) và 30 epochs tuyến tính tăng liều lượng rho lên 0.2
- Tự động trích xuất điểm Validation NDCG@20 tốt nhất và nạp checkpoint để đánh giá Full-Ranking Test đối chiếu V4 và V8


In [ ]:
# Cell 6a: Training STAIR5-v8.1 BSCV on Amazon Sports (Pha A: Pilot)
sports_time = run_training_stair5_v8_1('sports')
ep_sports, metrics_sports = extract_test_metrics(STAIR5_V8_1_CONFIGS['sports']['log'], STAIR5_V8_1_CONFIGS['sports']['artifact_dir'])
val_ep_sports, val_m_sports = extract_best_validation(STAIR5_V8_1_CONFIGS['sports']['log'])
STAIR5_V8_1_RESULTS['sports'] = {
    'metrics': metrics_sports,
    'best_epoch': ep_sports,
    'val_metrics': val_m_sports,
    'train_time': sports_time,
}

print('=' * 80)
print(f'🏆 KẾT QUẢ NGHIỆM THU AMAZON SPORTS (STAIR5-v8.1 BSCV / Best Epoch: {ep_sports}):')
for m, v in metrics_sports.items():
    v4_val = V4_REF['sports'].get(m, 0.0)
    v8_val = V8_REF['sports'].get(m, 0.0)
    delta_v4 = ((v - v4_val) / v4_val) * 100 if v4_val else 0.0
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    print(f'  * {m:12s}: {v:.4f} (v8: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}% | v4: {v4_val:.4f} | Δ vs v4: {delta_v4:+.2f}%)')
print('=' * 80)


## Cell 6b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Sports
- Vẽ đồ thị phân tích chi tiết mức tiêu thụ VRAM (Paper Standard) với đường trần lý thuyết (< 240 MiB Flat Memory Budget)


In [ ]:
# Cell 6b: Model Tensor VRAM Profile — Amazon Sports (Paper Standard)
_sports_art = STAIR5_V8_1_CONFIGS['sports']['artifact_dir']
_telemetry_sports = parse_telemetry_jsonl(_sports_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_sports:
    _eps = [r['epoch'] for r in _telemetry_sports]
    _vram_mb = [r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_sports]
    plt.plot(_eps, _vram_mb, color='#ff7f0e', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.axhline(240, color='red', ls=':', lw=1.5, label='Theoretical Ceiling Budget (240 MiB)')
    plt.title('Amazon Sports — Peak Model Tensor VRAM Profile (STAIR5-v8.1 BSCV)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'sports' in vram_profile and len(vram_profile['sports']) > 0:
    plt.plot(vram_profile['sports'], color='#ff7f0e', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Sports — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    print('Chưa có dữ liệu VRAM telemetry cho Amazon Sports.')
plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_sports.png', dpi=150)
plt.show()


## Cell 6c 📈 Động Lực Học, Quá Trình Hội Tụ & Consensus Diagnostics — Amazon Sports
- Trực quan hóa hệ thống hàm mất mát (Total Loss, BPR Loss, NLGCL Loss) song song với chất lượng xếp hạng Validation NDCG@20 trên trục kép
- Phân tích động thái liều lượng $\rho_e$ và giá trị trung bình cổng thỏa thuận `gate_mean` từ `consensus_telemetry.jsonl`


In [ ]:
# Cell 6c: Learning Dynamics & Convergence Profiles — Amazon Sports
_sports_art = STAIR5_V8_1_CONFIGS['sports']['artifact_dir']
_telemetry_sports = parse_telemetry_jsonl(_sports_art)
_consensus_sports = parse_consensus_telemetry_jsonl(_sports_art)
_log_sports = STAIR5_V8_1_CONFIGS['sports']['log']
_bpr_l, _cl_l, _tot_l = parse_training_losses(_log_sports, _telemetry_sports)
_val_ndcg = parse_valid_metric(_log_sports, 'NDCG@20')

_fig, (_ax1, _ax3) = plt.subplots(2, 1, figsize=(10, 8), dpi=150, sharex=True)

# Subplot 1: Losses & Validation NDCG@20
if _tot_l:
    _x_tot, _y_tot = zip(*_tot_l)
    _ax1.plot(_x_tot, _y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if _bpr_l:
    _x_bpr, _y_bpr = zip(*_bpr_l)
    _ax1.plot(_x_bpr, _y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

_ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
_ax1.tick_params(axis='y', labelcolor='#1f77b4')
_ax1.grid(True, linestyle='--', alpha=0.4)
_ax1.legend(loc='upper left', frameon=True)

if _val_ndcg:
    _ax2 = _ax1.twinx()
    _x_v, _y_v = zip(*_val_ndcg)
    _ax2.plot(_x_v, _y_v, color='#d62728', lw=2.0, marker='s', ms=4, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    _ax2.tick_params(axis='y', labelcolor='#d62728')
    _ax2.legend(loc='upper right', frameon=True)

_ax1.set_title('Amazon Sports — Learning Dynamics & Convergence Profiles (STAIR5-v8.1)', fontsize=12, fontweight='bold')

# Subplot 2: Consensus Dose & Gate Activation
if _consensus_sports:
    _c_eps = [r['epoch'] for r in _consensus_sports]
    _c_rho = [r.get('rho', 0.0) for r in _consensus_sports]
    _c_gate = [r.get('gate_mean', 0.5) for r in _consensus_sports if r.get('gate_mean') is not None]
    _ax3.plot(_c_eps, _c_rho, color='#2ca02c', lw=2.0, label='Consensus Dose ρ_e (Warmup 30 + Ramp 30)')
    _ax3.set_ylabel('Consensus Dose ρ', color='#2ca02c', fontsize=10)
    _ax3.tick_params(axis='y', labelcolor='#2ca02c')
    _ax3.grid(True, linestyle='--', alpha=0.4)
    if _c_gate and len(_c_gate) == len(_c_eps):
        _ax4 = _ax3.twinx()
        _ax4.plot(_c_eps, _c_gate, color='#9467bd', lw=1.8, ls='-.', label='Mean Gate Activation g_i')
        _ax4.set_ylabel('Gate Value g_i', color='#9467bd', fontsize=10)
        _ax4.tick_params(axis='y', labelcolor='#9467bd')
        _ax4.legend(loc='lower right', frameon=True)
    _ax3.legend(loc='upper left', frameon=True)
else:
    _ax3.text(0.5, 0.5, 'Consensus Telemetry Đang Được Ghi Nhận...', ha='center', va='center', transform=_ax3.transAxes)

_ax3.set_xlabel('Epoch', fontsize=10)
_ax3.set_title('Amazon Sports — Consensus Dose Schedule & Gate Agreement Dynamics', fontsize=11, fontweight='bold')

_fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_sports.png', dpi=150)
plt.show()


## Cell 7a 🏋️ Huấn luyện Pha B — Amazon Baby (Kiểm chứng An toàn STAIR5-v8.1 / BSCV)
- Huấn luyện mô hình trên tập dữ liệu Amazon Baby (~160K tương tác) kiểm chứng tính ổn định của toán tử BSCV
- Xác minh khả năng giữ vững hiệu năng xếp hạng trên dữ liệu thưa và ngắn hạn


In [ ]:
# Cell 7a: Training STAIR5-v8.1 BSCV on Amazon Baby (Pha B: Confirmatory)
baby_time = run_training_stair5_v8_1('baby')
ep_baby, metrics_baby = extract_test_metrics(STAIR5_V8_1_CONFIGS['baby']['log'], STAIR5_V8_1_CONFIGS['baby']['artifact_dir'])
val_ep_baby, val_m_baby = extract_best_validation(STAIR5_V8_1_CONFIGS['baby']['log'])
STAIR5_V8_1_RESULTS['baby'] = {
    'metrics': metrics_baby,
    'best_epoch': ep_baby,
    'val_metrics': val_m_baby,
    'train_time': baby_time,
}

print('=' * 80)
print(f'🏆 KẾT QUẢ NGHIỆM THU AMAZON BABY (STAIR5-v8.1 BSCV / Best Epoch: {ep_baby}):')
for m, v in metrics_baby.items():
    v4_val = V4_REF['baby'].get(m, 0.0)
    v8_val = V8_REF['baby'].get(m, 0.0)
    delta_v4 = ((v - v4_val) / v4_val) * 100 if v4_val else 0.0
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    print(f'  * {m:12s}: {v:.4f} (v8: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}% | v4: {v4_val:.4f} | Δ vs v4: {delta_v4:+.2f}%)')
print('=' * 80)


## Cell 7b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Baby
- Phân tích chi tiết mức tiêu thụ VRAM thực tế đối chiếu đường trần an toàn lý thuyết (< 220 MiB Flat Memory Budget)


In [ ]:
# Cell 7b: Model Tensor VRAM Profile — Amazon Baby (Paper Standard)
_baby_art = STAIR5_V8_1_CONFIGS['baby']['artifact_dir']
_telemetry_baby = parse_telemetry_jsonl(_baby_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_baby:
    _eps = [r['epoch'] for r in _telemetry_baby]
    _vram_mb = [r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_baby]
    plt.plot(_eps, _vram_mb, color='#1f77b4', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.axhline(220, color='red', ls=':', lw=1.5, label='Theoretical Ceiling Budget (220 MiB)')
    plt.title('Amazon Baby — Peak Model Tensor VRAM Profile (STAIR5-v8.1 BSCV)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'baby' in vram_profile and len(vram_profile['baby']) > 0:
    plt.plot(vram_profile['baby'], color='#1f77b4', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Baby — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    print('Chưa có dữ liệu VRAM telemetry cho Amazon Baby.')
plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_baby.png', dpi=150)
plt.show()


## Cell 7c 📈 Động Lực Học, Quá Trình Hội Tụ & Consensus Diagnostics — Amazon Baby
- Trực quan hóa đường cong hàm mất mát huấn luyện và chất lượng đánh giá Validation NDCG@20 song song với Consensus Telemetry


In [ ]:
# Cell 7c: Learning Dynamics & Convergence Profiles — Amazon Baby
_baby_art = STAIR5_V8_1_CONFIGS['baby']['artifact_dir']
_telemetry_baby = parse_telemetry_jsonl(_baby_art)
_consensus_baby = parse_consensus_telemetry_jsonl(_baby_art)
_log_baby = STAIR5_V8_1_CONFIGS['baby']['log']
_bpr_l, _cl_l, _tot_l = parse_training_losses(_log_baby, _telemetry_baby)
_val_ndcg = parse_valid_metric(_log_baby, 'NDCG@20')

_fig, (_ax1, _ax3) = plt.subplots(2, 1, figsize=(10, 8), dpi=150, sharex=True)

if _tot_l:
    _x_tot, _y_tot = zip(*_tot_l)
    _ax1.plot(_x_tot, _y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if _bpr_l:
    _x_bpr, _y_bpr = zip(*_bpr_l)
    _ax1.plot(_x_bpr, _y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

_ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
_ax1.tick_params(axis='y', labelcolor='#1f77b4')
_ax1.grid(True, linestyle='--', alpha=0.4)
_ax1.legend(loc='upper left', frameon=True)

if _val_ndcg:
    _ax2 = _ax1.twinx()
    _x_v, _y_v = zip(*_val_ndcg)
    _ax2.plot(_x_v, _y_v, color='#d62728', lw=2.0, marker='s', ms=4, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    _ax2.tick_params(axis='y', labelcolor='#d62728')
    _ax2.legend(loc='upper right', frameon=True)

_ax1.set_title('Amazon Baby — Learning Dynamics & Convergence Profiles (STAIR5-v8.1)', fontsize=12, fontweight='bold')

if _consensus_baby:
    _c_eps = [r['epoch'] for r in _consensus_baby]
    _c_rho = [r.get('rho', 0.0) for r in _consensus_baby]
    _c_gate = [r.get('gate_mean', 0.5) for r in _consensus_baby if r.get('gate_mean') is not None]
    _ax3.plot(_c_eps, _c_rho, color='#2ca02c', lw=2.0, label='Consensus Dose ρ_e')
    _ax3.set_ylabel('Consensus Dose ρ', color='#2ca02c', fontsize=10)
    _ax3.tick_params(axis='y', labelcolor='#2ca02c')
    _ax3.grid(True, linestyle='--', alpha=0.4)
    if _c_gate and len(_c_gate) == len(_c_eps):
        _ax4 = _ax3.twinx()
        _ax4.plot(_c_eps, _c_gate, color='#9467bd', lw=1.8, ls='-.', label='Mean Gate Activation g_i')
        _ax4.set_ylabel('Gate Value g_i', color='#9467bd', fontsize=10)
        _ax4.tick_params(axis='y', labelcolor='#9467bd')
        _ax4.legend(loc='lower right', frameon=True)
    _ax3.legend(loc='upper left', frameon=True)
else:
    _ax3.text(0.5, 0.5, 'Consensus Telemetry Đang Được Ghi Nhận...', ha='center', va='center', transform=_ax3.transAxes)

_ax3.set_xlabel('Epoch', fontsize=10)
_ax3.set_title('Amazon Baby — Consensus Dose Schedule & Gate Agreement Dynamics', fontsize=11, fontweight='bold')

_fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_baby.png', dpi=150)
plt.show()


## Cell 8a 🏋️ Huấn luyện Pha C — Amazon Electronics (~1.7M Tương tác, Scale-up STAIR5-v8.1)
- Huấn luyện mô hình quy mô lớn trên Amazon Electronics (~1.68 triệu tương tác, 192K Users, 63K Items)
- Kiểm chứng khả năng kháng OOM nhờ phân mảnh khối `cf_block_size=64`, `cl_chunk_size=128` (hoặc 1024 trên Blackwell)
- Khẳng định tính ổn định của đồ thị sparse CSR soft-consensus $C$ ở quy mô lớn


In [ ]:
# Cell 8a: Training STAIR5-v8.1 BSCV on Amazon Electronics (Pha C: Scale-up)
electronics_time = run_training_stair5_v8_1('electronics')
ep_elec, metrics_elec = extract_test_metrics(STAIR5_V8_1_CONFIGS['electronics']['log'], STAIR5_V8_1_CONFIGS['electronics']['artifact_dir'])
val_ep_elec, val_m_elec = extract_best_validation(STAIR5_V8_1_CONFIGS['electronics']['log'])
STAIR5_V8_1_RESULTS['electronics'] = {
    'metrics': metrics_elec,
    'best_epoch': ep_elec,
    'val_metrics': val_m_elec,
    'train_time': electronics_time,
}

print('=' * 80)
print(f'🏆 KẾT QUẢ NGHIỆM THU AMAZON ELECTRONICS (STAIR5-v8.1 BSCV / Best Epoch: {ep_elec}):')
for m, v in metrics_elec.items():
    v4_val = V4_REF['electronics'].get(m, 0.0)
    v8_val = V8_REF['electronics'].get(m, 0.0)
    delta_v4 = ((v - v4_val) / v4_val) * 100 if v4_val else 0.0
    delta_v8 = ((v - v8_val) / v8_val) * 100 if v8_val else 0.0
    print(f'  * {m:12s}: {v:.4f} (v8: {v8_val:.4f} | Δ vs v8: {delta_v8:+.2f}% | v4: {v4_val:.4f} | Δ vs v4: {delta_v4:+.2f}%)')
print('=' * 80)


## Cell 8b ⚡ Biểu đồ Tiêu thụ Bộ nhớ Tensor Mô hình — Amazon Electronics (Cổng An toàn Phần cứng)
- Phân tích chi tiết mức tiêu thụ VRAM đối chiếu với Cổng An toàn Phần cứng Nghiêm ngặt (< 800 MiB Strict Hardware Gate)


In [ ]:
# Cell 8b: Model Tensor VRAM Profile — Amazon Electronics (Memory Gate Verification)
_elec_art = STAIR5_V8_1_CONFIGS['electronics']['artifact_dir']
_telemetry_elec = parse_telemetry_jsonl(_elec_art)

plt.figure(figsize=(9, 4.5), dpi=150)
if _telemetry_elec:
    _eps = [r['epoch'] for r in _telemetry_elec]
    _vram_mb = [r.get('max_memory_allocated_bytes', 0) / (1024**2) for r in _telemetry_elec]
    plt.plot(_eps, _vram_mb, color='#2ca02c', lw=2.2, marker='o', ms=4, label='Peak Allocated VRAM (MB)')
    plt.axhline(800, color='red', ls=':', lw=1.5, label='Strict Memory Gate Ceiling (800 MiB)')
    plt.title('Amazon Electronics — Peak Model Tensor VRAM Profile (STAIR5-v8.1)', fontsize=12, fontweight='bold')
    plt.xlabel('Epoch', fontsize=10)
    plt.ylabel('Allocated VRAM (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
elif 'electronics' in vram_profile and len(vram_profile['electronics']) > 0:
    plt.plot(vram_profile['electronics'], color='#2ca02c', lw=1.8, label='Host NVML GPU Used (MB)')
    plt.title('Amazon Electronics — Host GPU Memory Profile (NVML)', fontsize=12, fontweight='bold')
    plt.xlabel('Polling Step (~2s)', fontsize=10)
    plt.ylabel('Memory (MB)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend(frameon=True)
else:
    print('Chưa có dữ liệu VRAM telemetry cho Amazon Electronics.')
plt.tight_layout()
plt.savefig('/kaggle/working/reports/vram_profile_electronics.png', dpi=150)
plt.show()


## Cell 8c 📈 Động Lực Học, Quá Trình Hội Tụ & Consensus Diagnostics — Amazon Electronics
- Trực quan hóa động lực học hàm mất mát và chất lượng xếp hạng Validation NDCG@20 trên tập dữ liệu quy mô lớn


In [ ]:
# Cell 8c: Learning Dynamics & Convergence Profiles — Amazon Electronics
_elec_art = STAIR5_V8_1_CONFIGS['electronics']['artifact_dir']
_telemetry_elec = parse_telemetry_jsonl(_elec_art)
_consensus_elec = parse_consensus_telemetry_jsonl(_elec_art)
_log_elec = STAIR5_V8_1_CONFIGS['electronics']['log']
_bpr_l, _cl_l, _tot_l = parse_training_losses(_log_elec, _telemetry_elec)
_val_ndcg = parse_valid_metric(_log_elec, 'NDCG@20')

_fig, (_ax1, _ax3) = plt.subplots(2, 1, figsize=(10, 8), dpi=150, sharex=True)

if _tot_l:
    _x_tot, _y_tot = zip(*_tot_l)
    _ax1.plot(_x_tot, _y_tot, color='#1f77b4', lw=2.0, label='Total Loss (BPR + λ·NLGCL)')
if _bpr_l:
    _x_bpr, _y_bpr = zip(*_bpr_l)
    _ax1.plot(_x_bpr, _y_bpr, color='#1f77b4', lw=1.5, ls='--', alpha=0.7, label='BPR Loss')

_ax1.set_ylabel('Training Loss', color='#1f77b4', fontsize=10)
_ax1.tick_params(axis='y', labelcolor='#1f77b4')
_ax1.grid(True, linestyle='--', alpha=0.4)
_ax1.legend(loc='upper left', frameon=True)

if _val_ndcg:
    _ax2 = _ax1.twinx()
    _x_v, _y_v = zip(*_val_ndcg)
    _ax2.plot(_x_v, _y_v, color='#d62728', lw=2.0, marker='s', ms=4, label='Validation NDCG@20')
    _ax2.set_ylabel('Validation NDCG@20', color='#d62728', fontsize=10)
    _ax2.tick_params(axis='y', labelcolor='#d62728')
    _ax2.legend(loc='upper right', frameon=True)

_ax1.set_title('Amazon Electronics — Learning Dynamics & Convergence Profiles (STAIR5-v8.1)', fontsize=12, fontweight='bold')

if _consensus_elec:
    _c_eps = [r['epoch'] for r in _consensus_elec]
    _c_rho = [r.get('rho', 0.0) for r in _consensus_elec]
    _c_gate = [r.get('gate_mean', 0.5) for r in _consensus_elec if r.get('gate_mean') is not None]
    _ax3.plot(_c_eps, _c_rho, color='#2ca02c', lw=2.0, label='Consensus Dose ρ_e')
    _ax3.set_ylabel('Consensus Dose ρ', color='#2ca02c', fontsize=10)
    _ax3.tick_params(axis='y', labelcolor='#2ca02c')
    _ax3.grid(True, linestyle='--', alpha=0.4)
    if _c_gate and len(_c_gate) == len(_c_eps):
        _ax4 = _ax3.twinx()
        _ax4.plot(_c_eps, _c_gate, color='#9467bd', lw=1.8, ls='-.', label='Mean Gate Activation g_i')
        _ax4.set_ylabel('Gate Value g_i', color='#9467bd', fontsize=10)
        _ax4.tick_params(axis='y', labelcolor='#9467bd')
        _ax4.legend(loc='lower right', frameon=True)
    _ax3.legend(loc='upper left', frameon=True)
else:
    _ax3.text(0.5, 0.5, 'Consensus Telemetry Đang Được Ghi Nhận...', ha='center', va='center', transform=_ax3.transAxes)

_ax3.set_xlabel('Epoch', fontsize=10)
_ax3.set_title('Amazon Electronics — Consensus Dose Schedule & Gate Agreement Dynamics', fontsize=11, fontweight='bold')

_fig.tight_layout()
plt.savefig('/kaggle/working/reports/learning_curve_electronics.png', dpi=150)
plt.show()


## Cell 9 📊 Bảng Đối chuẩn Đa thế hệ Toàn diện (Baseline vs v4 vs v6 vs v7 vs v8 vs v8.1)
- Bảng tổng hợp đối sánh hiệu năng xếp hạng trên cả 3 tập dữ liệu giữa tất cả các thế hệ STAIR
- Tính toán chính xác độ chênh lệch phần trăm ($\Delta$ vs Baseline, $\Delta$ vs v4, $\Delta$ vs v8)
- Xuất file báo cáo cấu trúc `reports/stair5_v8_1_benchmark_summary.csv` sẵn sàng cho khóa luận tốt nghiệp


In [ ]:
# Cell 9: Bảng Đối chuẩn Đa thế hệ Toàn diện (STAIR Baseline vs v4 vs v6 vs v7 vs v8 vs v8.1)
import pandas as pd
from prettytable import PrettyTable

_table = PrettyTable()
_table.field_names = ['Dataset', 'Kiến trúc / Phiên bản', *TRACKED_METRICS, 'Δ vs Baseline (%)', 'Δ vs v4 (%)', 'Δ vs v8 (%)']

_df_rows = []

for _ds in ('sports', 'baby', 'electronics'):
    _ref_bl = BASELINE_REF[_ds]['NDCG@20']
    _ref_v4 = V4_REF[_ds]['NDCG@20']
    _ref_v6 = V6_REF[_ds]['NDCG@20']
    _ref_v7 = V7_REF[_ds]['NDCG@20']
    _ref_v8 = V8_REF[_ds]['NDCG@20']

    # 1. Baseline
    _table.add_row([_ds, 'STAIR (baseline)', *[f"{BASELINE_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], '-', '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR (baseline)', **{_m: BASELINE_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': 0.0, 'Δ vs v4': 0.0, 'Δ vs v8': 0.0})

    # 2. v4
    _d_v4_bl = 100 * (_ref_v4 - _ref_bl) / _ref_bl
    _table.add_row([_ds, 'STAIR5-v4 (NLGCL-CSE)', *[f"{V4_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v4_bl:+.2f}%", '-', '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v4', **{_m: V4_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v4_bl, 'Δ vs v4': 0.0, 'Δ vs v8': 0.0})

    # 3. v6
    _d_v6_bl = 100 * (_ref_v6 - _ref_bl) / _ref_bl
    _d_v6_v4 = 100 * (_ref_v6 - _ref_v4) / _ref_v4
    _table.add_row([_ds, 'STAIR5-v6 (UCR)', *[f"{V6_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v6_bl:+.2f}%", f"{_d_v6_v4:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v6', **{_m: V6_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v6_bl, 'Δ vs v4': _d_v6_v4, 'Δ vs v8': 0.0})

    # 4. v7
    _d_v7_bl = 100 * (_ref_v7 - _ref_bl) / _ref_bl
    _d_v7_v4 = 100 * (_ref_v7 - _ref_v4) / _ref_v4
    _table.add_row([_ds, 'STAIR5-v7 (UCR-D)', *[f"{V7_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v7_bl:+.2f}%", f"{_d_v7_v4:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v7', **{_m: V7_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v7_bl, 'Δ vs v4': _d_v7_v4, 'Δ vs v8': 0.0})

    # 5. v8
    _d_v8_bl = 100 * (_ref_v8 - _ref_bl) / _ref_bl
    _d_v8_v4 = 100 * (_ref_v8 - _ref_v4) / _ref_v4
    _table.add_row([_ds, 'STAIR5-v8 (WMSG-CSE)', *[f"{V8_REF[_ds][_m]:.4f}" for _m in TRACKED_METRICS], f"{_d_v8_bl:+.2f}%", f"{_d_v8_v4:+.2f}%", '-'])
    _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v8 (WMSG-CSE)', **{_m: V8_REF[_ds][_m] for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v8_bl, 'Δ vs v4': _d_v8_v4, 'Δ vs v8': 0.0})

    # 6. v8.1 (BSCV)
    if _ds in STAIR5_V8_1_RESULTS and STAIR5_V8_1_RESULTS[_ds].get('metrics'):
        _v81_m = STAIR5_V8_1_RESULTS[_ds]['metrics']
        _ndcg_v81 = _v81_m.get('NDCG@20', 0.0)
        _d_v81_bl = 100 * (_ndcg_v81 - _ref_bl) / _ref_bl
        _d_v81_v4 = 100 * (_ndcg_v81 - _ref_v4) / _ref_v4
        _d_v81_v8 = 100 * (_ndcg_v81 - _ref_v8) / _ref_v8
        _table.add_row([
            _ds, 'STAIR5-v8.1 (BSCV)',
            *[f"{_v81_m.get(_m, 0.0):.4f}" for _m in TRACKED_METRICS],
            f"{_d_v81_bl:+.2f}%", f"{_d_v81_v4:+.2f}%", f"{_d_v81_v8:+.2f}%"
        ])
        _df_rows.append({'Dataset': _ds, 'Model': 'STAIR5-v8.1 (BSCV)', **{_m: _v81_m.get(_m, 0.0) for _m in TRACKED_METRICS}, 'Δ vs Baseline': _d_v81_bl, 'Δ vs v4': _d_v81_v4, 'Δ vs v8': _d_v81_v8})

print('=' * 110)
print('📊 BẢNG TỔNG HỢP SO SÁNH HIỆU NĂNG ĐA THẾ HỆ (Baseline vs v4 vs v6 vs v7 vs v8 vs v8.1):')
print('=' * 110)
print(_table)

# Xuất ra file CSV
_df_summary = pd.DataFrame(_df_rows)
os.makedirs('/kaggle/working/reports', exist_ok=True)
_summary_csv = '/kaggle/working/reports/stair5_v8_1_benchmark_summary.csv'
_df_summary.to_csv(_summary_csv, index=False)
print(f'\n✅ Báo cáo tổng hợp đã lưu tại: {_summary_csv}')


## Cell 10 📦 Đóng gói Lưu trữ Toàn bộ Minh chứng, Logs, Manifests & Checkpoints
- Nén toàn bộ Artifacts của STAIR5-v8.1 thành định dạng `.tar.gz` và `.zip`
- Đặt tại `/kaggle/working/stair5_v81_artifacts.tar.gz` sẵn sàng để tải về trực tiếp từ giao diện Kaggle Output


In [ ]:
# Cell 10: Đóng gói Lưu trữ Toàn bộ Minh chứng (Logs, Checkpoints, Telemetry, Reports)
import os, shutil, json, time

_archive_dir = '/kaggle/working/stair5_v81_artifacts'
os.makedirs(_archive_dir, exist_ok=True)

# 1. Thu thập logs, reports và runs
_target_dirs = [
    '/kaggle/working/logs/stair5_v81',
    '/kaggle/working/reports',
    '/kaggle/working/runs/stair5_v81',
    os.path.expanduser('~/logs/stair5_v81'),
    os.path.expanduser('~/reports'),
]
for _src in _target_dirs:
    if os.path.exists(_src):
        _dst = os.path.join(_archive_dir, os.path.basename(_src))
        try:
            shutil.copytree(_src, _dst, dirs_exist_ok=True)
        except Exception:
            pass

# 2. Tạo manifest tổng thể cho phiên thử nghiệm
_manifest_data = {
    'run_id': RUN_ID,
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'framework': 'STAIR5-v8.1 (BSCV)',
    'results': {k: v.get('metrics', {}) for k, v in STAIR5_V8_1_RESULTS.items()},
    'configs': {k: {cfg_k: str(cfg_v) for cfg_k, cfg_v in v.items()} for k, v in STAIR5_V8_1_CONFIGS.items()},
}
with open(os.path.join(_archive_dir, 'run_manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(_manifest_data, f, indent=2, ensure_ascii=False)

# 3. Nén thành archive
_tar_path = '/kaggle/working/stair5_v81_artifacts.tar.gz'
_zip_path = '/kaggle/working/stair5_v81_artifacts.zip'
print('📦 Đang đóng gói toàn bộ thư mục artifacts...')
shutil.make_archive('/kaggle/working/stair5_v81_artifacts', 'gztar', _archive_dir)
shutil.make_archive('/kaggle/working/stair5_v81_artifacts', 'zip', _archive_dir)

print('=' * 80)
print('✅ QUÁ TRÌNH THỰC NGHIỆM STAIR5-v8.1 (BSCV) HOÀN TẤT THÀNH CÔNG!')
print(f'  * File lưu trữ Tar.gz: {_tar_path} ({os.path.getsize(_tar_path)/(1024**2):.2f} MB)')
print(f'  * File lưu trữ Zip   : {_zip_path} ({os.path.getsize(_zip_path)/(1024**2):.2f} MB)')
print('=' * 80)
